We were on the right track here, but not quite.

<!-- $$
z_{V}\equiv\frac{h_{\text{visitor}}+\text{offset}_{V}}{m_{V}}, \quad
\qquad m_{V}=\mathrm{median}(h_{\text{visitor}})
$$

Suppose 

h = 0
offset = 0

expect hydraulically active area goes to zero before estuary depth goes to zero. 

 -->

In [15]:
# --- Imports & config ---
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize_scalar
from scipy.optimize import least_squares  

# Data search path
data_paths = [Path('pooled_event_df.csv'), Path('../data/processed/pooled_event_df.csv')]
data_path = next((p for p in data_paths if p.exists()), data_paths[-1])
print('Using data:', data_path)

# --- Load data ---
df = pd.read_csv(data_path)
# Required numeric columns
for col in ["delta_h", "seepage"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["visitor_h"] = pd.to_numeric(df["visitor_h"], errors="coerce")


Using data: ../data/processed/pooled_event_df.csv


In [16]:
# small numeric constant
eps = 1e-12

# drop rows missing the required regressors/target
df = df.dropna(subset=["delta_h", "seepage", 'visitor_h']).copy()

# per-event row counts (used for row-weighted summaries)
df["n"] = df.groupby("event")["event"].transform("size").astype("int64")
# df = df.query("n >= 5")

print(f"rows : {len(df):,} | events: {df['event'].nunique():,}")

rows : 443 | events: 36


In [25]:
# --- global normalization for estuary level (h_est) ---
mask_h_est = df[["delta_h", "seepage", "visitor_h"]].notna().all(axis=1)
h_est = df.loc[mask_h_est, "visitor_h"].to_numpy(float)


# single normalizer used in both pooled & grouped fits
h_ref = float(np.median(h_est))  
print("Median visitor center water level:", round(h_ref, 6))

# --- global offset bounds for h_est  ---
offset_lb_h_est = - float(h_est.min()) + 1e-9  # ensures h_est + offset >= 0

q05, q95 = np.quantile(h_est, [0.05, 0.95])
spread = float(max(q95 - q05, 1e-6))
site_cap_h_est = 2
offset_ub_h_est = min( spread, site_cap_h_est)

print("h_est offset bounds (global):", (round(offset_lb_h_est, 2), round(offset_ub_h_est, 2)))


Median visitor center water level: 2.644743
h_est offset bounds (global): (-1.6, 1.4)


In [26]:
df.visitor_h.min()

1.59921575

In [54]:
import numpy as np
from scipy.optimize import least_squares

def r2_centered(y, yhat):
    """
    R^2: 1 - SSE / SST, with SST computed around the sample mean of y.
    Use centered R^2 so comparisons are fair across different closures/means.
    """
    y = np.asarray(y, float); yhat = np.asarray(yhat, float)
    sst = np.sum((y - np.mean(y))**2)
    if sst <= 0:
        return np.nan
    sse = np.sum((y - yhat)**2)
    return 1.0 - sse / sst

def weighted_median(values, weights):
    """
    Row-weighted (n-weighted) median: longer closures count more in grouped summaries,
    matching a row-weighted (micro-SSE) pooled objective.
    """
    v = np.asarray(values, float)
    w = np.asarray(weights, float)
    ok = np.isfinite(v) & np.isfinite(w) & (w > 0)
    v, w = v[ok], w[ok]
    if len(v) == 0:
        return np.nan
    order = np.argsort(v)
    v, w = v[order], w[order]
    cw = np.cumsum(w) / np.sum(w)
    return float(v[np.searchsorted(cw, 0.5)])

# model forms
def model_linear(k, dh):
    """Linear (Δh): S = k * Δh"""
    return k * dh

def model_visitor_norm(k, off, b, dh, h_est_arr, h_ref):
    """
    Visitor-normalized model:
        S = k * Δh * ((h_est + off) / h_rev)^b
    One global h_rev_ is used everywhere (pooled & grouped) to keep k interpretable.
    """
    base = np.maximum(h_est_arr + off, eps)
    print (off)
    return k * dh * np.power((h_est_arr + off) /h_ref, b)

# residual maps for joint least_squares
def residuals_linear_joint(params, dh, y):
    """
    Joint residuals for linear model. params = [k]
    """
    k = float(params[0])
    return y - model_linear(k, dh)

def residuals_visitor_joint(params, dh, h_est_arr, y, h_rev_):
    """
    Joint residuals for visitor-normalized model. params = [k, off, b]
    Bounds will be enforced via least_squares bounds argument.
    """
    k, off, b = map(float, params)
    yhat = model_visitor_norm(k, off, b, dh, h_est_arr, h_rev_)
    return y - yhat


In [64]:
# ---------- pooled fits on all rows (SSE-only, minimize_scalar) ----------

# assumes these exist already from earlier steps:
#   eps, h_ref, offset_lb_h_est, offset_ub_h_est
# and r2_centered(), model_visitor_norm(), model_linear() are defined (see previous sections)

import numpy as np
from scipy.optimize import minimize_scalar

# # SSE at fixed (off, b); k solved in closed form
def sse_for_off_b(off, b, dh, h_est_arr, y, h_ref_):
    if (off < offset_lb_h_est) or (off > offset_ub_h_est) or (b < 0):
        return np.inf, None, None
    z = dh * np.power(np.maximum(h_est_arr + off, eps) / max(h_ref_, eps), b)
    denom = max(np.dot(z, z), eps)
    k = max(float(np.dot(z, y) / denom), 1e-12)
    resid = y - k * z
    return float(np.sum(resid * resid)), k, z

# # 1-D minimize in b for a given off (grid start + bounded refine)
def best_b_for_off(off, dh, h_est_arr, y, h_ref_, b_lo=1e-6, b_hi=2.0, grid_n=41):
    # coarse scan
    b_grid = np.linspace(b_lo, b_hi, grid_n)
    sse_vals = []
    for b in b_grid:
        sse_b, _, _ = sse_for_off_b(off, b, dh, h_est_arr, y, h_ref_)
        sse_vals.append(sse_b)
    b_init = float(b_grid[int(np.argmin(sse_vals))])

    # local refine around the best coarse point
    left  = max(b_lo, b_init - 0.5)
    right = min(b_hi, b_init + 0.5)

    # objective wrapper for minimize_scalar (depends only on b)
    def sse_over_b(b):
        return sse_for_off_b(off, b, dh, h_est_arr, y, h_ref_)[0]

    res_b = minimize_scalar(sse_over_b, bounds=(left, right), method="bounded", options={"xatol": 1e-6})
    b_star = float(res_b.x)
    sse_star, k_star, _  = sse_for_off_b(off, b_star, dh, h_est_arr, y, h_ref_)
    return b_star, k_star, sse_star

# # outer 1-D minimize in off (grid start + bounded refine); returns (k*, off*, b*, sse*)
def profile_offset_all_rows_min_scalar(dh, h_est_arr, y, h_ref_):
    lo, hi = (offset_lb_h_est + 1e-9), (offset_ub_h_est - 1e-9)

    # coarse scan in off
    off_grid = np.linspace(lo, hi, 41)
    sse_list = []
    for off in off_grid:
        b_hat, k_hat, sse_hat = best_b_for_off(off, dh, h_est_arr, y, h_ref_)
        sse_list.append(sse_hat)
    off_init = float(off_grid[int(np.argmin(sse_list))])

    # local refine in off
    span = 0.25 * (hi - lo)
    left  = max(lo, off_init - span)
    right = min(hi, off_init + span)

    def sse_over_off(off):
        return best_b_for_off(off, dh, h_est_arr, y, h_ref_)[2]

    res_off = minimize_scalar(sse_over_off, bounds=(left, right), method="bounded", options={"xatol": 1e-6})
    off_star = float(res_off.x)

    # finalize b*, k* at off*
    b_star, k_star, sse_star = best_b_for_off(off_star, dh, h_est_arr, y, h_ref_)
    return k_star, off_star, b_star, sse_star

# linear (Δh) — closed-form pooled fit
def pooled_linear_delta_h(df_in):
    dh = df_in["delta_h"].to_numpy(float)
    y  = df_in["seepage"].to_numpy(float)
    denom = max(np.dot(dh, dh), eps)
    k = max(float(np.dot(dh, y) / denom), 1e-12)
    yhat = model_linear(k, dh)
    return {"k": k, "r2": r2_centered(y, yhat), "n": len(dh)}

# visitor (normalized) — pooled fit via 1-D off and 1-D b (k in closed form)
def pooled_visitor_min_scalar(df_in, h_ref_):
    dh = df_in["delta_h"].to_numpy(float)
    h_arr = df_in["visitor_h"].to_numpy(float)
    y  = df_in["seepage"].to_numpy(float)

    k_star, off_star, b_star, _ = profile_offset_all_rows_min_scalar(dh, h_arr, y, h_ref_)
    yhat = model_visitor_norm(k_star, off_star, b_star, dh, h_arr, h_ref_)

    return {"k": k_star, "offset": off_star, "b": b_star, "r2": r2_centered(y, yhat), "n": len(y)}

# run pooled fits (all rows)
pooled_lin = pooled_linear_delta_h(df)
pooled_vis = pooled_visitor_min_scalar(df, h_ref)

# readable printout
print("\npooled (linear Δh):",
      {k: round(v, 3) if isinstance(v, (int, float)) else v for k, v in pooled_lin.items()})
print("pooled (visitor model, minimize_scalar):",
      {k: round(v, 3) if isinstance(v, (int, float)) else v for k, v in pooled_vis.items()})

# simple bound diagnostic (offset on bound?)
hit_lb = np.isclose(pooled_vis["offset"], offset_lb_h_est, rtol=0, atol=1e-6)
hit_ub = np.isclose(pooled_vis["offset"], offset_ub_h_est, rtol=0, atol=1e-6)
if hit_lb or hit_ub:
    which = "lower" if hit_lb else "upper"
    bound = offset_lb_h_est if hit_lb else offset_ub_h_est
#     print(f"[warn] visitor pooled offset is on the {which} bound ({round(bound, 6)}). "
#           "Consider relaxing the cap slightly if this is unintended.")


1.398002875685202

pooled (linear Δh): {'k': 1.069, 'r2': 0.39, 'n': 443}
pooled (visitor model, minimize_scalar): {'k': 0.775, 'offset': 1.398, 'b': 0.681, 'r2': 0.407, 'n': 443}


In [65]:
# ---------- config derived from the data ----------
eps = 1e-12

# visitor model normalizer (global)
v_ref = float(df["visitor_h"].median())

# global offset bounds for visitor_h
q05, q95 = np.quantile(df["visitor_h"], [0.05, 0.95])
spread = float(max(q95 - q05, 1e-6))
offset_lb_vis = -float(df["visitor_h"].min()) + 1e-9      # ensures visitor_h + offset >= 0
offset_ub_vis = min( spread,2)                    # gentle site-scale cap

# ---------- small utilities (top-level) ----------
def r2_centered(y, yhat):
    y = np.asarray(y, float); yhat = np.asarray(yhat, float)
    sst = np.sum((y - np.mean(y))**2)
    if sst <= 0:
        return np.nan
    sse = np.sum((y - yhat)**2)
    return 1.0 - sse / sst

def weighted_median(values, weights):
    v = np.asarray(values, float)
    w = np.asarray(weights, float)
    ok = np.isfinite(v) & np.isfinite(w) & (w > 0)
    v, w = v[ok], w[ok]
    if len(v) == 0:
        return np.nan
    order = np.argsort(v)
    v, w = v[order], w[order]
    cw = np.cumsum(w) / np.sum(w)
    return float(v[np.searchsorted(cw, 0.5)])

# ---------- model forms ----------
def model_linear(k, dh):
    return k * dh

def model_visitor_norm(k, off, b, dh, vis, v_ref_):
    base = np.maximum(vis + off, eps)
    return k * dh * np.power(base / max(v_ref_ + off, eps), b)

# ---------- pooled linear (Δh), all rows ----------
def pooled_linear_delta_h(df_in):
    x = df_in["delta_h"].to_numpy(float)
    y = df_in["seepage"].to_numpy(float)
    denom = max(np.dot(x, x), eps)
    k = float(np.dot(x, y) / denom)
    k = max(k, 1e-12)
    yhat = model_linear(k, x)
    out = {"k": k, "r2": r2_centered(y, yhat), "n": len(x)}
    return out

# ---------- pooled visitor model (profile offset & b), all rows ----------
# for fixed (off, b): optimal k is closed-form -> k* = (z⋅y)/(z⋅z), z = dh*((vis+off)/v_ref)**b
def sse_for_off_b(off, b, dh, vis, y, v_ref_):
    if (off < offset_lb_vis) or (off > offset_ub_vis) or (b < 0):
        return np.inf, None
    z = dh * np.power(np.maximum(vis + off, eps) / max(v_ref_ + off, eps), b)
    denom = max(np.dot(z, z), eps) 
    k = max(float(np.dot(z, y) / denom), 1e-12)
    resid = y - k * z
    return float(np.sum(resid*resid)), k

def best_b_given_off(off, dh, vis, y, v_ref_):
    # 1-D search in b >= 0; coarse grid + local refine
    b_lo, b_hi = 1e-6, 2.0
    grid = np.linspace(b_lo, b_hi, 41)
    sse_vals = [sse_for_off_b(off, b, dh, vis, y, v_ref_)[0] for b in grid]
    b_init = float(grid[int(np.argmin(sse_vals))])
    def obj_b(b):
        return sse_for_off_b(off, b, dh, vis, y, v_ref_)[0]
    res_b = minimize_scalar(obj_b, bounds=(max(b_lo, b_init - 0.5), min(b_hi, b_init + 0.5)), method="bounded")
    b_star = float(res_b.x)
    sse_star, k_star = sse_for_off_b(off, b_star, dh, vis, y, v_ref_)
    return b_star, k_star, sse_star

def profile_offset_all_rows(dh, vis, y, v_ref_):
    lo, hi = (offset_lb_vis + 1e-9), (offset_ub_vis - 1e-9)
    grid = np.linspace(lo, hi, 41)
    # coarse pass
    scores = []
    for off in grid:
        b_hat, k_hat, sse_hat = best_b_given_off(off, dh, vis, y, v_ref_)
        scores.append(sse_hat)
    off_init = float(grid[int(np.argmin(scores))])
    # local refine in offset
    span = 0.25 * (hi - lo)
    left, right = max(lo, off_init - span), min(hi, off_init + span)
    def obj_off(off):
        return best_b_given_off(off, dh, vis, y, v_ref_)[2]
    res_off = minimize_scalar(obj_off, bounds=(left, right), method="bounded")
    off_star = float(res_off.x)
    b_star, k_star, sse_star = best_b_given_off(off_star, dh, vis, y, v_ref_)
    return k_star, off_star, b_star, sse_star

def pooled_visitor(df_in, v_ref_):
    dh = df_in["delta_h"].to_numpy(float)
    vis = df_in["visitor_h"].to_numpy(float)
    y  = df_in["seepage"].to_numpy(float)
    k, off, b, sse = profile_offset_all_rows(dh, vis, y, v_ref_)
    yhat = model_visitor_norm(k, off, b, dh, vis, v_ref_)
    out = {"k": k, "offset": off, "b": b, "r2": r2_centered(y, yhat), "n": len(y)}
    return out

# ---------- grouped per-event visitor model, k only with pooled (off, b) fixed ----------
def grouped_visitor_k_only(df_in, v_ref_, off_star, b_star):
    rows = []
    for ev, g in df_in.groupby("event"):
        dh = g["delta_h"].to_numpy(float)
        vis = g["visitor_h"].to_numpy(float)
        y   = g["seepage"].to_numpy(float)
        if len(dh) < 1:
            continue
        z = dh * np.power(np.maximum(vis + off_star, eps) / max(v_ref_ + off_star, eps), b_star)
        denom = max(np.dot(z, z), eps)
        k_e = max(float(np.dot(z, y) / denom), 1e-12)
        rows.append({"event": ev, "n": len(dh), "k": k_e})
    ev_tbl = pd.DataFrame(rows).sort_values("event").reset_index(drop=True)
    return ev_tbl

# ---------- run pooled fits on ALL rows ----------
pooled_lin = pooled_linear_delta_h(df)
pooled_vis = pooled_visitor(df, v_ref)

print("\npooled (linear Δh):", {k: round(v, 3) if isinstance(v, (float, int)) else v for k, v in pooled_lin.items()})
print("pooled (visitor model):", {k: round(v, 3) if isinstance(v, (float, int)) else v for k, v in pooled_vis.items()})

# ---------- run grouped: k only for visitor (offset,b fixed to pooled); and linear k per event ----------
# linear (Δh) grouped k_e
ev_lin_rows = []
for ev, g in df.groupby("event"):
    dh = g["delta_h"].to_numpy(float)
    y  = g["seepage"].to_numpy(float)
    if len(dh) < 1:
        continue
    denom = max(np.dot(dh, dh), eps)
    k_e = max(float(np.dot(dh, y) / denom), 1e-12)
    ev_lin_rows.append({"event": ev, "n": len(dh), "k": k_e})
ev_lin_tbl = pd.DataFrame(ev_lin_rows).sort_values("event").reset_index(drop=True)

# visitor (k only, fixed pooled (off,b))
ev_vis_tbl = grouped_visitor_k_only(df, v_ref, pooled_vis["offset"], pooled_vis["b"])

# ---------- n-weighted medians (row-weighted summaries) ----------
k_lin_wmed = weighted_median(ev_lin_tbl["k"], ev_lin_tbl["n"])
k_vis_wmed = weighted_median(ev_vis_tbl["k"], ev_vis_tbl["n"])

# ---------- display simple comparison ----------
summary_rows = [
    {"model": "linear (Δh)",
     "pooled_k": pooled_lin["k"],
     "grouped_k_wmed": k_lin_wmed,
     "n_rows": pooled_lin["n"]},
    {"model": "visitor (normalized)",
     "pooled_k": pooled_vis["k"],
     "pooled_offset": pooled_vis["offset"],
     "pooled_b": pooled_vis["b"],
     "grouped_k_wmed": k_vis_wmed,
     "n_rows": pooled_vis["n"],
     "v_ref": v_ref}
]
summary = pd.DataFrame(summary_rows)
for col in summary.select_dtypes(include=[float]).columns:
    summary[col] = summary[col].round(3)
print("\ncomparison (pooled vs grouped n-weighted medians):")
print(summary.to_string(index=False))



pooled (linear Δh): {'k': 1.069, 'r2': 0.39, 'n': 443}
pooled (visitor model): {'k': 1.035, 'offset': 1.398, 'b': 0.681, 'r2': 0.407, 'n': 443}

comparison (pooled vs grouped n-weighted medians):
               model  pooled_k  grouped_k_wmed  n_rows  pooled_offset  pooled_b  v_ref
         linear (Δh)     1.069           1.018     443            NaN       NaN    NaN
visitor (normalized)     1.035           0.950     443          1.398     0.681  2.645
